- GPU：**Accelerator = GPU T4 x2**（免费档也可能 roll 到 **P100**，都行）—— 模型**全 fp32**（fp16 模型会冻结训练，见配置节），distogram einsum 走 fp16 tensor core（T4 加速；P100 无 TC 只是不加速、不影响正确性）

In [ ]:
# ① 安装依赖（版本钉死：与保存 checkpoint 的环境一致 TF 2.21 + Keras 3.15）
# Kaggle 在 Google Cloud，默认 PyPI 直连即可（不要用国内镜像，反而慢/失败）
!pip install -q "tensorflow==2.21" "keras==3.15" datasets huggingface_hub pyarrow polars pyyaml tqdm tensorboard

import tensorflow as tf
import keras
print("TensorFlow:", tf.__version__, "| Keras:", keras.__version__)

In [ ]:
# ② 代码就位：代码直接放 /kaggle/working 下（编辑器上传的文件会随 Commit 一起跑），例如：
#      /kaggle/working/spice_pre/...           （模型代码包）
#      /kaggle/working/configs/pretrain.yaml   （或 /kaggle/working/config/pretrain.yaml）
#    若代码在只读的挂载 Dataset（/kaggle/input/...），本 cell 会自动拷到 /kaggle/working/model。
#    本 cell 自动识别，无需改动（自己的上传/挂载代码可以贴在最前面）。

import os, sys, shutil, glob

WORK = "/kaggle/working"

def has_code(d):
    return os.path.isdir(os.path.join(d, "spice_pre"))

# 情形 A：代码直接在 /kaggle/working 下 —— 不拷贝，直接用
if has_code(WORK):
    CODE_ROOT = WORK
    print("✅ 代码直接在 /kaggle/working 下（无需拷贝）")
else:
    # 情形 B：代码在只读挂载的 Dataset → 拷到 /kaggle/working/model（可写、结果持久）
    SRC = None
    for pat in ["/kaggle/input/*/model", "/kaggle/input/*/*/model", "/kaggle/input/*"]:
        for h in sorted(glob.glob(pat)):
            if has_code(h):
                SRC = h
                break
        if SRC:
            break
    if SRC is None:
        print("❌ 没找到 spice_pre 代码。请把代码传到 /kaggle/working/spice_pre，或挂载含 model/ 的 Dataset。")
        raise SystemExit(1)
    CODE_ROOT = "/kaggle/working/model"
    os.makedirs(CODE_ROOT, exist_ok=True)
    for name in os.listdir(SRC):
        s = os.path.join(SRC, name)
        d = os.path.join(CODE_ROOT, name)
        if os.path.isdir(s):
            shutil.copytree(s, d, dirs_exist_ok=True)
        else:
            shutil.copy2(s, d)
    print(f"✅ 已把代码从 {SRC} 拷到 {CODE_ROOT}")

sys.path.insert(0, CODE_ROOT)
os.chdir(CODE_ROOT)   # 相对输出路径（data/ runs/ checkpoints/）落在 /kaggle/working → 自动保存
print("cwd =", os.getcwd())
print("spice_pre OK:", has_code(CODE_ROOT))

## 配置
- **🔴 精度（2026-08-13 决定性实验）**：**`use_mixed_precision=false`（模型全 fp32）+ `distogram_fp16=true`（einsum 走 tensor core）**。fp16 混合精度会把模型"冻住"（梯度在深层 fp16 backprop 下溢到 0 → 30 epoch 跑完 rmsd 卡 150Å、CE 卡 52）；E 组合（fp32 模型 + fp16 einsum）本地验证收链 2.3Å。P100/T4 都适用
- **🔴 长度门控（2026-08-13 决定性）**：**`coord_max_len: 200`**——frame 坐标损失只给 ≤200aa 的链；长链（200-400aa 占全数据 65%）cumsum 误差累积会毒化 encoder，但**距离任务在长链上可学** → distogram 全 45k 学、坐标监督只给短链。`max_seq_len: 512`（不丢数据）
- **⚠️ 判断标准 = 看 `ce`，别看 `rmsd`**：门控后长链坐标不监督 → **`rmsd` 会一直高（几十到几百 Å，正常！）**。成败看 `ce`：必须一路跌破 ~4（随机 ln48≈3.87）并持续降到 2-3。**别用 rmsd 判断，别中途杀**（10h GPU 预算输不起）
- **HF 端点 = 官方 huggingface.co**（覆盖 yaml 的 hf-mirror.com —— Kaggle 在 Google Cloud 直连官方，`_set_hf_endpoint` 非 Colab 时读 `cfg.data.hf_endpoint`，必须覆盖）
- **单卡**：`gpu_devices="0"`（落盘 cache 与 MirroredStrategy 冲突，小模型吃不下 2 卡；P100 只有 1 卡）
- **落盘 cache**：`data/train_cache` 先建后训，消除中途 cache 构建假象 + RAM OOM
- fp32 模型比 fp16 慢 ~1.5-2×：fp16 4.2h → fp32 约 5.5h（30 epoch，10h 预算内）

In [ ]:
# ③ 配置加载 + 环境自检（HF 官方端点、fp32 模型 + fp16 einsum、单卡、版本）
import os
os.environ["HF_ENDPOINT"] = "https://huggingface.co"          # Google Cloud 直连官方（覆盖国内镜像）

from spice_pre.config import load_config
from spice_pre.keras_utils import setup_gpu

# 兼容 configs/ 与 config/ 两种目录名
cfg_path = next((p for p in ["configs/pretrain.yaml", "config/pretrain.yaml"] if os.path.exists(p)), None)
assert cfg_path, "没找到 pretrain.yaml（configs/ 或 config/）"

cfg = load_config(cfg_path)
cfg.data.hf_endpoint = "https://huggingface.co"               # 双保险：_set_hf_endpoint 非 Colab 时读它

# 🔴 fp16 混合精度会冻结训练（2026-08-13 决定性实验，E/F/G 网格）：
#    梯度在深层 fp16 backprop 下溢到 0（grad_norm=0，30 epoch 跑完 rmsd 卡 150Å、distogram CE 卡 52）。
#    修复 = 模型全 fp32：
cfg.train.use_mixed_precision = False
#    distogram einsum 仍走 fp16（tensor core 加速）——fp32 模型 + fp16 einsum 本地已验证收链 2.3Å，安全：
cfg.model.distogram_fp16 = True

setup_gpu(cfg.train.use_gpu, cfg.train.gpu_mem_growth, cfg.train.gpu_devices)

import tensorflow as tf
import keras
gpus = tf.config.list_physical_devices("GPU")


def _gpu_names():
    try:
        import subprocess
        out = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True, timeout=10).stdout
        return out.upper().strip()
    except Exception:
        return ""


print("TensorFlow:", tf.__version__, "| Keras:", keras.__version__)
print("GPU:", "ON" if cfg.train.use_gpu else "OFF (forced CPU)")
print("GPU devices:", gpus if gpus else ["CPU"])
print("GPU detail:", _gpu_names() or "n/a")
print("mixed_precision:", cfg.train.use_mixed_precision, "(强制 false：fp16 模型会冻结训练)")
print("distogram_fp16:", cfg.model.distogram_fp16, "(einsum 走 fp16 tensor core，安全)")
print("HF endpoint:", os.environ.get("HF_ENDPOINT"))
print("config:", cfg_path)
print("train: batch=%s epochs=%s max_steps=%s lr=%s" % (cfg.train.batch_size, cfg.train.epochs, cfg.train.max_steps, cfg.train.lr))
print("frame: dist=%s pair=%s warmup=%s chi=%s clash=%s" % (
    cfg.train.dist_weight, cfg.train.pair_weight, cfg.train.pair_warmup_steps,
    cfg.train.frame_chirality_weight, cfg.train.frame_clash_weight))

In [ ]:
# ④ 构建 TFRecord（全 shard ~1.4GB 下载；Commit 一次跑完，无需人工介入）
from spice_pre.data.dataset import build_tfrecords

cfg.data.max_shards = 0
cfg.data.use_env_filtered = False

n = build_tfrecords(cfg)
print("TFRecord record count:", n)

In [ ]:
# ⑤ 预构建落盘 cache（幂等；改 batch/分桶后需删 data/train_cache.* 重建）
from spice_pre.train_pretrain import build_train_cache

build_train_cache(cfg)

In [ ]:
# ⑥ 训练 30 epoch（Commit 上限 12h；预计 4~6h 完成）
from spice_pre.train_pretrain import train

cfg.train.epochs = 30
cfg.train.max_steps = 0

train(cfg)

## 跑完怎么拿结果
- **Output 标签**：Commit 跑完自动保存，打开本 notebook → 右栏 Output，可下载整个 `/kaggle/working`（20GB 内）
- 关键产物：
  - `checkpoints/pretrain/best_weights.weights.h5`（新 frame 头权重，替换本地后喂 RL）
  - `runs/pretrain/`（tensorboard 曲线 + CSV）
  - `data/tfrecords/`、`data/train_cache.*`（缓存，下次可挂 Input 复用省下载）
- 也可把本 notebook 的 Output 当 **Input 挂到下一个 notebook**（Data Source → Notebooks）
- 训练曲线本地看：`tensorboard --logdir runs/pretrain`